In [2]:
import numpy as np
import pandas as pd

### Timestamp Object

Time stamps reference particular moments in time (e.g., Oct 24th, 2022 at 7:00pm)

### Creating Timestamp objects

## 🔬 Deep Dive : Timestamp objects

```python
type(pd.Timestamp('2023/1/5'))       # pandas._libs.tslibs.timestamps.Timestamp
pd.Timestamp('2023-1-5') / '2023, 1, 5' / '2023' / '5th January 2023' / '5th January 2023 9:21AM'
pd.Timestamp(dt.datetime(2023,1,5))  # from python datetime
x.year; x.month; x.day; x.hour; x.minute; x.second   # attribute access
```

- Timestamp = single moment in time (enhanced datetime)
- Parses MANY flexible string formats automatically
- AM/PM, ordinal ('5th'), text months all work
- Attributes: year, month, day, hour, minute, second, microsecond
- Construct from datetime, np.datetime64, string, epoch
- **Why pandas not datetime?**: performance + timezone + vector ops
  - datetime is convenient but slow in loops
  - pandas ops vectorized
- Foundation of all time series work
- `.to_pydatetime()` conversion back
- comparisons supported like datetime

In [4]:
# creating a timestamp
type(pd.Timestamp('2023/1/5'))

pandas._libs.tslibs.timestamps.Timestamp

In [6]:
# variations
pd.Timestamp('2023-1-5')
pd.Timestamp('2023, 1, 5')

Timestamp('2023-01-05 00:00:00')

In [7]:
# only year
pd.Timestamp('2023')

Timestamp('2023-01-01 00:00:00')

In [8]:
# using text
pd.Timestamp('5th January 2023')

Timestamp('2023-01-05 00:00:00')

In [16]:
# providing time also
pd.Timestamp('5th January 2023 9:21AM')
# pd.Timestamp('2023/1/5/9/21')

ValueError: ignored

In [65]:
# AM and PM

In [20]:
# using datetime.datetime object
import datetime as dt

x = pd.Timestamp(dt.datetime(2023,1,5,9,21,56))
x

Timestamp('2023-01-05 09:21:56')

In [26]:
# fetching attributes
x.year
x.month
x.day
x.hour
x.minute
x.second

56

## 🔬 Deep Dive : numpy datetime64 & why vectorized time

```python
date = np.array('2015-07-04', dtype=np.datetime64)
date + np.arange(12)     # all 12 next days at once!

# uniform dtype → fast arithmetic
# python datetime list would need a loop
```

- np.datetime64 — numpy's time dtype (uniform, fast)
- Addition with ints = days shift vectorized
- Whole arrays shifted together — no python loop
- Reason: performance + uniform memory
- pandas builds on numpy → its own Timestamp layer
- pandas DatetimeIndex ops similarly vectorized
- Yearly/monthly/day arithmetic natural
- 64-bit int backing — compact memory
- This is performance foundation of DateTime in pandas

In [ ]:
# why separate objects to handle data and time when python already has datetime functionality?

- syntax wise datetime is very convenient
- But the performance takes a hit while working with huge data. List vs Numpy Array
- The weaknesses of Python's datetime format inspired the NumPy team to add a set of native time series data type to NumPy.
- The datetime64 dtype encodes dates as 64-bit integers, and thus allows arrays of dates to be represented very compactly.

In [44]:
import numpy as np
date = np.array('2015-07-04', dtype=np.datetime64)
date

array('2015-07-04', dtype='datetime64[D]')

In [45]:
date + np.arange(12)

array(['2015-07-04', '2015-07-05', '2015-07-06', '2015-07-07',
       '2015-07-08', '2015-07-09', '2015-07-10', '2015-07-11',
       '2015-07-12', '2015-07-13', '2015-07-14', '2015-07-15'],
      dtype='datetime64[D]')

- Because of the uniform type in NumPy datetime64 arrays, this type of operation can be accomplished much more quickly than if we were working directly with Python's datetime objects, especially as arrays get large 

- Pandas Timestamp object combines the ease-of-use of python datetime with the efficient storage and vectorized interface of numpy.datetime64

- From a group of these Timestamp objects, Pandas can construct a DatetimeIndex that can be used to index data in a Series or DataFrame

### DatetimeIndex Object

A collection of pandas timestamp

## 🔬 Deep Dive : DatetimeIndex

```python
pd.DatetimeIndex(['2023/1/1','2022/1/1','2021/1/1'])
pd.DatetimeIndex([dt.datetime(2023,1,1), dt.datetime(2020,1,1)])
pd.Series([1,2,3], index=pd.DatetimeIndex([pd.Timestamp(2023,1,1), ...]))
```

- DatetimeIndex = collection of Timestamps → used as INDEX
- Built from strings / datetime objects / Timestamps
- Series indexed by dates → time-indexed data
- Enables time-slicing, resampling, plotting with dates on x-axis
- can infer index from parseable column
- date_sorting matters (chronological indexing)
- `.index` on time-indexed df gives DatetimeIndex
- Advance slicing: yt.loc['2022-12'] partial-indexing works
- Time-based lookup instant after sort

In [32]:
# from strings
type(pd.DatetimeIndex(['2023/1/1','2022/1/1','2021/1/1']))

pandas.core.indexes.datetimes.DatetimeIndex

In [33]:
# using python datetime object
pd.DatetimeIndex([dt.datetime(2023,1,1),dt.datetime(2022,1,1),dt.datetime(2021,1,1)])

DatetimeIndex(['2023-01-01', '2022-01-01', '2021-01-01'], dtype='datetime64[ns]', freq=None)

In [37]:
# using pd.timestamps
dt_index = pd.DatetimeIndex([pd.Timestamp(2023,1,1),pd.Timestamp(2022,1,1),pd.Timestamp(2021,1,1)])

In [39]:
# using datatimeindex as series index

pd.Series([1,2,3],index=dt_index)

2023-01-01    1
2022-01-01    2
2021-01-01    3
dtype: int64

### date_range function



## 🔬 Deep Dive : date_range — generate date sequences

```python
pd.date_range(start='2023/1/5', end='2023/2/28')
pd.date_range(start='2023/1/5', end='2023/2/28', freq='B')    # business days
pd.date_range(start='2023/1/5', end='2023/2/28', freq='H')    # hourly
pd.date_range(start='2023/1/5', end='2023/2/28', freq='MS')   # month start
pd.date_range(start='2023/1/5', periods=25)                   # by count
```

- freq codes:
  - D daily, B business, W weekly, H hourly
  - M month-end, MS month-start, A year-end
  - combos: '2D','3H','W-MON','BM'
- start/end OR periods — both ways
- Creates DatetimeIndex — template for data gaps
- Calendar-aware calendars (business days skip weekends)
- Filling experiment dates
- Common: daily series a month, sub-hourly
- resample/asfreq rely on freq logic
- ISO calendar/weekday support — freq detailed
- Creating dates without data — simulation, joins

In [43]:
# generate daily dates in a given range
pd.date_range(start='2023/1/5',end='2023/2/28',freq='3D')

DatetimeIndex(['2023-01-05', '2023-01-08', '2023-01-11', '2023-01-14',
               '2023-01-17', '2023-01-20', '2023-01-23', '2023-01-26',
               '2023-01-29', '2023-02-01', '2023-02-04', '2023-02-07',
               '2023-02-10', '2023-02-13', '2023-02-16', '2023-02-19',
               '2023-02-22', '2023-02-25', '2023-02-28'],
              dtype='datetime64[ns]', freq='3D')

In [ ]:
# alternate days in a given range
pd.date_range(start='2023/1/5',end='2023/2/28',freq='3D')

In [44]:
# B -> business days
pd.date_range(start='2023/1/5',end='2023/2/28',freq='B')

DatetimeIndex(['2023-01-05', '2023-01-06', '2023-01-09', '2023-01-10',
               '2023-01-11', '2023-01-12', '2023-01-13', '2023-01-16',
               '2023-01-17', '2023-01-18', '2023-01-19', '2023-01-20',
               '2023-01-23', '2023-01-24', '2023-01-25', '2023-01-26',
               '2023-01-27', '2023-01-30', '2023-01-31', '2023-02-01',
               '2023-02-02', '2023-02-03', '2023-02-06', '2023-02-07',
               '2023-02-08', '2023-02-09', '2023-02-10', '2023-02-13',
               '2023-02-14', '2023-02-15', '2023-02-16', '2023-02-17',
               '2023-02-20', '2023-02-21', '2023-02-22', '2023-02-23',
               '2023-02-24', '2023-02-27', '2023-02-28'],
              dtype='datetime64[ns]', freq='B')

In [46]:
# W -> one week per day
pd.date_range(start='2023/1/5',end='2023/2/28',freq='W-THU')

DatetimeIndex(['2023-01-05', '2023-01-12', '2023-01-19', '2023-01-26',
               '2023-02-02', '2023-02-09', '2023-02-16', '2023-02-23'],
              dtype='datetime64[ns]', freq='W-THU')

In [48]:
# H -> Hourly data(factor)
pd.date_range(start='2023/1/5',end='2023/2/28',freq='6H')

DatetimeIndex(['2023-01-05 00:00:00', '2023-01-05 06:00:00',
               '2023-01-05 12:00:00', '2023-01-05 18:00:00',
               '2023-01-06 00:00:00', '2023-01-06 06:00:00',
               '2023-01-06 12:00:00', '2023-01-06 18:00:00',
               '2023-01-07 00:00:00', '2023-01-07 06:00:00',
               ...
               '2023-02-25 18:00:00', '2023-02-26 00:00:00',
               '2023-02-26 06:00:00', '2023-02-26 12:00:00',
               '2023-02-26 18:00:00', '2023-02-27 00:00:00',
               '2023-02-27 06:00:00', '2023-02-27 12:00:00',
               '2023-02-27 18:00:00', '2023-02-28 00:00:00'],
              dtype='datetime64[ns]', length=217, freq='6H')

In [49]:
# M -> Month end
pd.date_range(start='2023/1/5',end='2023/2/28',freq='M')

DatetimeIndex(['2023-01-31', '2023-02-28'], dtype='datetime64[ns]', freq='M')

In [50]:
# MS -> Month start
pd.date_range(start='2023/1/5',end='2023/2/28',freq='MS')

DatetimeIndex(['2023-02-01'], dtype='datetime64[ns]', freq='MS')

In [51]:
# A -> Year end
pd.date_range(start='2023/1/5',end='2030/2/28',freq='A')

DatetimeIndex(['2023-12-31', '2024-12-31', '2025-12-31', '2026-12-31',
               '2027-12-31', '2028-12-31', '2029-12-31'],
              dtype='datetime64[ns]', freq='A-DEC')

In [56]:
# using periods(number of results)
pd.date_range(start='2023/1/5',periods=25,freq='M')

DatetimeIndex(['2023-01-31', '2023-02-28', '2023-03-31', '2023-04-30',
               '2023-05-31', '2023-06-30', '2023-07-31', '2023-08-31',
               '2023-09-30', '2023-10-31', '2023-11-30', '2023-12-31',
               '2024-01-31', '2024-02-29', '2024-03-31', '2024-04-30',
               '2024-05-31', '2024-06-30', '2024-07-31', '2024-08-31',
               '2024-09-30', '2024-10-31', '2024-11-30', '2024-12-31',
               '2025-01-31'],
              dtype='datetime64[ns]', freq='M')

### to_datetime function

converts an existing objects to pandas timestamp/datetimeindex object

## 🔬 Deep Dive : to_datetime — string → Timestamp column

```python
s = pd.Series(['2023/1/1','2022/1/1','2021/1/1'])
pd.to_datetime(s)                          # parse whole column
pd.to_datetime(s, errors='coerce')         # bad dates → NaT (not crash)
```

- to_datetime — CONVERT existing series/df of strings to datetime
- errors='coerce' → unparseable become NaT (missing)
- errors='raise' default — crash on bad
- apply to df col: df['Date'] = pd.to_datetime(df['Date'])
- format= arg for custom formats (speed + correctness)
- yearfirst/dayfirst for ambiguous dates
- After conversion: dtypes datetime64[ns], info shows
- Then .dt accessor, plotting, resampling available
- Weekly pipeline: read CSV → to_datetime → index → analyze
- Datetime col = 80% of real datasets 'date' column

In [65]:
# simple series example

s = pd.Series(['2023/1/1','2022/1/1','2021/1/1'])
pd.to_datetime(s).dt.day_name()

0      Sunday
1    Saturday
2      Friday
dtype: object

In [70]:
# with errors
s = pd.Series(['2023/1/1','2022/1/1','2021/130/1'])
pd.to_datetime(s,errors='coerce').dt.month_name()

0    January
1    January
2        NaN
dtype: object

In [72]:
df = pd.read_csv('/content/expense_data.csv')
df.shape

(277, 11)

In [8]:
df.head()

             Date               Account        Category  Subcategory  \
0  3/2/2022 10:11  CUB - online payment            Food          NaN   
1  3/2/2022 10:11  CUB - online payment           Other          NaN   
2  3/1/2022 19:50  CUB - online payment            Food          NaN   
3  3/1/2022 18:56  CUB - online payment  Transportation          NaN   
4  3/1/2022 18:22  CUB - online payment            Food          NaN   

               Note    INR Income/Expense  Note.1  Amount Currency  Account.1  
0           Brownie   50.0        Expense     NaN    50.0      INR       50.0  
1  To lended people  300.0        Expense     NaN   300.0      INR      300.0  
2            Dinner   78.0        Expense     NaN    78.0      INR       78.0  
3             Metro   30.0        Expense     NaN    30.0      INR       30.0  
4            Snacks   67.0        Expense     NaN    67.0      INR       67.0  

In [75]:
df['Date'] = pd.to_datetime(df['Date'])

In [76]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 277 entries, 0 to 276
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Date            277 non-null    datetime64[ns]
 1   Account         277 non-null    object        
 2   Category        277 non-null    object        
 3   Subcategory     0 non-null      float64       
 4   Note            273 non-null    object        
 5   INR             277 non-null    float64       
 6   Income/Expense  277 non-null    object        
 7   Note.1          0 non-null      float64       
 8   Amount          277 non-null    float64       
 9   Currency        277 non-null    object        
 10  Account.1       277 non-null    float64       
dtypes: datetime64[ns](1), float64(5), object(5)
memory usage: 23.9+ KB


### dt accessor

Accessor object for datetimelike properties of the Series values.

In [83]:
df['Date'].dt.is_quarter_start

0      False
1      False
2      False
3      False
4      False
       ...  
272    False
273    False
274    False
275    False
276    False
Name: Date, Length: 277, dtype: bool

In [84]:
# plot graph
import matplotlib.pyplot as plt
plt.plot(df['Date'],df['INR'])

<Figure size 432x288 with 1 Axes>

## 🔬 Deep Dive : dt accessor + date plots

```python
df['day_name'] = df['Date'].dt.day_name()         # Monday..
df['month_name'] = df['Date'].dt.month_name()     # January..
df['Date'].dt.is_quarter_start / is_month_end     # bool flags
df.groupby('day_name')['INR'].mean().plot(kind='bar')
df.groupby('month_name')['INR'].sum().plot(kind='bar')
df[df['Date'].dt.is_month_end]                    # filter month-end dates
```

- .dt — accessor for datetime props (like .str for strings)
- day_name/month_name — easiest grouping labels
- is_month_start/end, is_quarter_start/end — period flags
- Extract: hour, weekday, quarter via .dt
- groupby extracted name → trend bars (avg per weekday)
- filter flags: month-end transactions
- expense analysis use-case driven
- datetime col → valuable insights here
- plot direct: plt.plot(df['Date'], df['INR'])
- Careful: timezone, locale names

In [86]:
# day name wise bar chart/month wise bar chart

df['day_name'] = df['Date'].dt.day_name()

In [87]:
df.head()

                 Date               Account        Category  Subcategory  \
0 2022-03-02 10:11:00  CUB - online payment            Food          NaN   
1 2022-03-02 10:11:00  CUB - online payment           Other          NaN   
2 2022-03-01 19:50:00  CUB - online payment            Food          NaN   
3 2022-03-01 18:56:00  CUB - online payment  Transportation          NaN   
4 2022-03-01 18:22:00  CUB - online payment            Food          NaN   

               Note    INR Income/Expense  Note.1  Amount Currency  Account.1  \
0           Brownie   50.0        Expense     NaN    50.0      INR       50.0   
1  To lended people  300.0        Expense     NaN   300.0      INR      300.0   
2            Dinner   78.0        Expense     NaN    78.0      INR       78.0   
3             Metro   30.0        Expense     NaN    30.0      INR       30.0   
4            Snacks   67.0        Expense     NaN    67.0      INR       67.0   

    day_name  
0  Wednesday  
1  Wednesday  
2    Tuesda

In [93]:
df.groupby('day_name')['INR'].mean().plot(kind='bar')

<Figure size 432x288 with 1 Axes>

In [90]:
df['month_name'] = df['Date'].dt.month_name()

In [91]:
df.head()

                 Date               Account        Category  Subcategory  \
0 2022-03-02 10:11:00  CUB - online payment            Food          NaN   
1 2022-03-02 10:11:00  CUB - online payment           Other          NaN   
2 2022-03-01 19:50:00  CUB - online payment            Food          NaN   
3 2022-03-01 18:56:00  CUB - online payment  Transportation          NaN   
4 2022-03-01 18:22:00  CUB - online payment            Food          NaN   

               Note    INR Income/Expense  Note.1  Amount Currency  Account.1  \
0           Brownie   50.0        Expense     NaN    50.0      INR       50.0   
1  To lended people  300.0        Expense     NaN   300.0      INR      300.0   
2            Dinner   78.0        Expense     NaN    78.0      INR       78.0   
3             Metro   30.0        Expense     NaN    30.0      INR       30.0   
4            Snacks   67.0        Expense     NaN    67.0      INR       67.0   

    day_name month_name  
0  Wednesday      March  
1  W

In [92]:
df.groupby('month_name')['INR'].sum().plot(kind='bar')

<Figure size 432x288 with 1 Axes>

In [95]:
df[df['Date'].dt.is_month_end]

                   Date               Account        Category  Subcategory  \
7   2022-02-28 11:56:00  CUB - online payment            Food          NaN   
8   2022-02-28 11:45:00  CUB - online payment           Other          NaN   
61  2022-01-31 08:44:00  CUB - online payment  Transportation          NaN   
62  2022-01-31 08:27:00  CUB - online payment           Other          NaN   
63  2022-01-31 08:26:00  CUB - online payment  Transportation          NaN   
242 2021-11-30 14:24:00  CUB - online payment            Gift          NaN   
243 2021-11-30 14:17:00  CUB - online payment            Food          NaN   
244 2021-11-30 10:11:00  CUB - online payment            Food          NaN   

                   Note     INR Income/Expense  Note.1  Amount Currency  \
7                 Pizza  339.15        Expense     NaN  339.15      INR   
8           From kumara  200.00         Income     NaN  200.00      INR   
61           Vnr to apk   50.00        Expense     NaN   50.00      INR 